# Refresh Download of Reviews

In [13]:
import requests
import pandas as pd


# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games[['appid', 'name']].head() #get first 5 rows of appid and name

,appid,name
730,730,Counter-Strike: Global Offensive
1172470,1172470,Apex Legends
578080,578080,PUBG: BATTLEGROUNDS
1623730,1623730,Palworld
440,440,Team Fortress 2


In [14]:
from dotenv import load_dotenv
from steam.webapi import WebAPI
import os

load_dotenv()

STEAM_API_KEY = os.getenv('STEAM_API_KEY')
api = WebAPI(key=STEAM_API_KEY)

# Example: Get reviews for a single app
def get_reviews(appid, num_reviews=100):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'filter': 'recent',
        'language': 'english'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return data['reviews']
    return []

In [ ]:
# Preview of CS review JSON data
cs_reviews = get_reviews(730, num_reviews=100)
cs_reviews[0]

{'recommendationid': '199023291',
 'author': {'steamid': '76561198033740077',
  'num_games_owned': 0,
  'num_reviews': 1,
  'playtime_forever': 3941,
  'playtime_last_two_weeks': 1344,
  'playtime_at_review': 3941,
  'last_played': 1751712743},
 'language': 'english',
 'review': 'Is ok xD',
 'timestamp_created': 1751727297,
 'timestamp_updated': 1751727297,
 'voted_up': True,
 'votes_up': 0,
 'votes_funny': 0,
 'weighted_vote_score': 0.5,
 'comment_count': 0,
 'steam_purchase': True,
 'received_for_free': False,
 'written_during_early_access': False,
 'primarily_steam_deck': False}

In [4]:
from tqdm import tqdm

all_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    appid = row['appid']
    name = row['name']
    reviews = get_reviews(appid, num_reviews=100)
    for review in reviews:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

  1%|          | 1/100 [00:01<02:37,  1.59s/it]


KeyboardInterrupt: 

In [ ]:
reviews_df.to_csv('reviews.csv', index=False)

# Start Here for analysis

In [1]:
import pandas as pd

df = pd.read_csv('reviews.csv')

In [3]:
df.describe(include='all')

,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
count,9.886000e+03,9886,9845,9.886000e+03,9886,9886.000000,9886.000000,9886.000000
unique,NaN,100,8775,NaN,2,NaN,NaN,NaN
top,NaN,Counter-Strike: Global Offensive,good,NaN,True,NaN,NaN,NaN
freq,NaN,100,127,NaN,7936,NaN,NaN,NaN
mean,7.154145e+05,NaN,NaN,1.726394e+09,NaN,1.018106,0.154259,0.502725
std,7.058122e+05,NaN,NaN,6.608987e+07,NaN,8.781307,1.419987,0.029328
min,1.000000e+01,NaN,NaN,1.453142e+09,NaN,0.000000,0.000000,0.203082
25%,2.389600e+05,NaN,NaN,1.747648e+09,NaN,0.000000,0.000000,0.500000
50%,4.381000e+05,NaN,NaN,1.751217e+09,NaN,0.000000,0.000000,0.500000
75%,1.091500e+06,NaN,NaN,1.751532e+09,NaN,0.000000,0.000000,0.500000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9886 entries, 0 to 9885
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   appid                9886 non-null   int64  
 1   name                 9886 non-null   object 
 2   review               9845 non-null   object 
 3   timestamp_created    9886 non-null   int64  
 4   voted_up             9886 non-null   bool   
 5   votes_up             9886 non-null   int64  
 6   votes_funny          9886 non-null   int64  
 7   weighted_vote_score  9886 non-null   float64
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 550.4+ KB


appid: The unique steam id for each game 
name: The unique game name
review: The corpus of all text in the review
timestamp_created: Unix time (epoch) in UTC (POSIX TIME) seconds since 01/01/1970
`voted_up`: The reviewer's thumb up (True) or thumb down (False) Score `(our target)`
votes_up: Number of people who upvoted the review
votes_funny: Number of people who thought the vote was funny
